# CEFR Multi-Prefix Tuning — 537M PMT-Matched Variant

This notebook trains and evaluates the **~537M CEFR Multi-Prefix Tuning model** used in the thesis.

The experiment is designed as a parameter-matched comparison with the Vanilla PrefixMemory-Tuning (PMT) baseline.

The model uses a frozen `meta-llama/Llama-3.1-8B-Instruct` backbone and six CEFR-specific prefix banks corresponding to A1, A2, B1, B2, C1, and C2.

Each CEFR level contains 30 learned virtual prefix tokens. The selected prefix embeddings are processed by a high-capacity shared projection network:

`4096 → 7696 → 65536`

The resulting controller contains **536,698,384 trainable parameters**, closely matching the **536,870,912** trainable parameters of Vanilla PMT.

The notebook includes:

- construction of the PMT-matched CEFR Multi-Prefix controller;
- parameter-count verification;
- training on the Balanced CEFR Steering Subset;
- validation-loss and perplexity tracking;
- hardware profiling;
- publication of the trained controller to Hugging Face;
- generation on the final In-Domain Evaluation Prompt Matrix;
- evaluation using the primary Joint-Loss CEFR evaluator.

The final model is used to separate the effect of **explicit CEFR conditioning** from the effect of simply increasing controller capacity.

In [ ]:
# =========================================================
# 0. ENVIRONMENT SETUP & DEPENDENCIES
# =========================================================
!pip install -q transformers peft torch datasets huggingface_hub accelerate scikit-learn textstat spacy hf_transfer "torchao>=0.16.0" nvidia-ml-py
!python -m spacy download en_core_web_sm

import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

import gc
import json
import math
import time
import threading
import pynvml
import torch
import torch.nn as nn
import pandas as pd
import numpy as np

from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    get_cosine_schedule_with_warmup,
    set_seed
)

from transformers.cache_utils import DynamicCache
from huggingface_hub import HfApi, login
from google.colab import userdata
from tqdm.auto import tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.1/177.1 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 94.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 107.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 102.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 134.4 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


/usr/local/lib/python3.13/dist-packages/huggingface_hub/constants.py:299: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(


In [ ]:
# =========================================================
# 1. EXPERIMENT IDENTIFIERS / OUTPUT PATHS
# =========================================================

EXPERIMENT_NAME = "cefr_prefix_tuning_parameter_matched_pmt_537m"

# ---------------------------------------------------------
# Dataset & Google Drive paths
# ---------------------------------------------------------
BALANCED_CSV_PATH = "/content/drive/MyDrive/Your_Path/"
    "balanced_cefr_steering_subset.csv"

DRIVE_LOG_DIR = (
    "/content/drive/MyDrive/Your_Path/"
    "cefr_prefix_tuning_537m_logs"
)

DRIVE_CHECKPOINT_DIR = (
    "/content/drive/MyDrive/Your_Path/"
    "cefr_prefix_tuning_537m_checkpoints"
)

# ---------------------------------------------------------
# Local output directory
# ---------------------------------------------------------
LOCAL_OUTPUT_DIR = "/content/cefr_pt_parameter_matched_537m_output"

os.makedirs(DRIVE_LOG_DIR, exist_ok=True)
os.makedirs(DRIVE_CHECKPOINT_DIR, exist_ok=True)
os.makedirs(LOCAL_OUTPUT_DIR, exist_ok=True)

In [ ]:
# =========================================================
# 2. HUGGING FACE REPOSITORY & AUTHENTICATION
# =========================================================
HF_REPO_ID = "MohammadKhosravi/llama3.1-8b-cefr-pt-537m-param-matched"

try:
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token)
except Exception:
    pass

# =========================================================
# 3. REPRODUCIBILITY & DEVICE
# =========================================================
SEED = 42
set_seed(SEED)
torch.backends.cudnn.deterministic = True

device = "cuda" if torch.cuda.is_available() else "cpu"

print("=" * 70)
print("EXPERIMENT:", EXPERIMENT_NAME)
print("HF REPOSITORY:", HF_REPO_ID)
print(f"DEVICE: {device}")
print("=" * 70)

EXPERIMENT: cefr_prefix_tuning_parameter_matched_pmt_537m
HF REPOSITORY: MohammadKhosravi/llama3.1-8b-cefr-pt-537m-param-matched
DEVICE: cuda


In [ ]:
import os
import shutil
from google.colab import drive
mountpoint = "/content/drive"

if os.path.exists(mountpoint):
    shutil.rmtree(mountpoint)

drive.mount(mountpoint)

Mounted at /content/drive


In [ ]:
# =========================================================
# 4. MOUNT GOOGLE DRIVE
# =========================================================

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# =========================================================
# 5. INGEST BALANCED DATASET (6k STRATIFIED SPLIT)
# =========================================================
print(f"\n📥 Ingesting Training Dataset from: {BALANCED_CSV_PATH}")

df = pd.read_csv(BALANCED_CSV_PATH).dropna(
    subset=["cefr", "clean_text", "topic_title"]
)
df["cefr"] = df["cefr"].str.upper().str.strip()

label_map = {"A1": 0, "A2": 1, "B1": 2, "B2": 3, "C1": 4, "C2": 5}
df["cefr_id"] = df["cefr"].map(label_map)

df = df.dropna(subset=["cefr_id"])
df["cefr_id"] = df["cefr_id"].astype(int)

train_df, val_df = train_test_split(
    df,
    test_size=0.10,
    stratify=df["cefr_id"],
    random_state=42
)

print(f"Training partition size:   {len(train_df):,} samples")
print(f"Validation partition size: {len(val_df):,} samples")


📥 Ingesting Training Dataset from: /content/drive/MyDrive/Mohammd_Thesis/subsets/efcamdat_balanced_subset_training.csv
Training partition size:   5,011 samples
Validation partition size: 557 samples


In [ ]:
# =========================================================
# 6. LOAD FROZEN BASE LLM & TOKENIZER
# =========================================================
model_id = "meta-llama/Llama-3.1-8B-Instruct"
print(f"\nLoading tokenizer and base model: {model_id}...")

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.padding_side = "right"

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map={"": torch.cuda.current_device()}
)

# Freeze entire Llama backbone
for param in base_model.parameters():
    param.requires_grad = False

base_model.eval()
base_model.config.use_cache = False


Loading tokenizer and base model: meta-llama/Llama-3.1-8B-Instruct...


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

In [ ]:
# =========================================================
# 7. PARAMETER-MATCHED CEFR MULTI-PREFIX CONTROLLER
# =========================================================
class CEFRMultiPrefixController(nn.Module):
    """
    Parameter-matched CEFR Prefix-Tuning controller.
    Six CEFR levels each have an independent set of 30 virtual prefix embeddings.
    A shared MLP transforms the selected prefix embeddings into layer-wise key/value states.
    This version increases MLP capacity to approximately match the 536.87M trainable parameters of vanilla PMT.
    """
    def __init__(
        self,
        config,
        num_virtual_tokens=30,
        num_classes=6,
        mlp_hidden_dim=7696
    ):
        super().__init__()
        self.num_virtual_tokens = num_virtual_tokens
        self.num_classes = num_classes
        self.num_layers = config.num_hidden_layers
        self.hidden_size = config.hidden_size
        self.num_kv_heads = config.num_key_value_heads
        self.head_dim = getattr(config, "head_dim", config.hidden_size // config.num_attention_heads)
        self.mlp_hidden_dim = mlp_hidden_dim

        # 6 CEFR levels × 30 virtual tokens
        self.prefix_embeddings = nn.Embedding(
            num_classes * num_virtual_tokens,
            self.hidden_size
        )

        flat_out_dim = self.num_layers * 2 * self.num_kv_heads * self.head_dim

        # HIGH-CAPACITY SHARED MLP (4096 -> 7696 -> 65536)
        self.prefix_mlp = nn.Sequential(
            nn.Linear(self.hidden_size, self.mlp_hidden_dim),
            nn.Tanh(),
            nn.Linear(self.mlp_hidden_dim, flat_out_dim)
        )

    def forward(self, cefr_ids):
        batch_size = cefr_ids.shape[0]

        # Select the 30 prefix embeddings associated with the requested CEFR level
        class_offsets = (cefr_ids * self.num_virtual_tokens).unsqueeze(1)
        base_indices = torch.arange(self.num_virtual_tokens, device=cefr_ids.device).unsqueeze(0)
        token_indices = class_offsets + base_indices

        prefix_tokens = self.prefix_embeddings(token_indices) # [B, 30, 4096]
        past_kv_flat = self.prefix_mlp(prefix_tokens)         # [B, 30, 65536]

        # Reshape to Llama GQA KV format
        past_kv = past_kv_flat.view(
            batch_size,
            self.num_virtual_tokens,
            self.num_layers,
            2,
            self.num_kv_heads,
            self.head_dim
        )

        # Permute to: [layers, 2, B, kv_heads, 30, head_dim]
        past_kv = past_kv.permute(2, 3, 0, 4, 1, 5)

        past_key_values = DynamicCache()
        for i in range(self.num_layers):
            past_key_values.update(past_kv[i, 0], past_kv[i, 1], layer_idx=i)

        return past_key_values

In [ ]:
# =========================================================
# 8. INITIALIZE CONTROLLER
# =========================================================
MLP_HIDDEN_DIM = 7696

prefix_controller = CEFRMultiPrefixController(
    base_model.config,
    num_virtual_tokens=30,
    num_classes=6,
    mlp_hidden_dim=MLP_HIDDEN_DIM
).to(
    device=device,
    dtype=torch.bfloat16
)

In [ ]:
# =========================================================
# 9. PARAMETER COUNT VERIFICATION
# =========================================================
trainable_params = sum(p.numel() for p in prefix_controller.parameters() if p.requires_grad)
total_controller_params = sum(p.numel() for p in prefix_controller.parameters())

PMT_REFERENCE_PARAMS = 536_870_912
parameter_difference = trainable_params - PMT_REFERENCE_PARAMS
parameter_difference_pct = (parameter_difference / PMT_REFERENCE_PARAMS) * 100

print("\n" + "=" * 70)
print("🔢 PARAMETER-MATCHING VERIFICATION")
print("=" * 70)
print(f"CEFR PT trainable parameters : {trainable_params:,} ({trainable_params / 1e6:.3f}M)")
print(f"Vanilla PMT reference        : {PMT_REFERENCE_PARAMS:,} ({PMT_REFERENCE_PARAMS / 1e6:.3f}M)")
print(f"Difference                   : {parameter_difference:+,} ({parameter_difference_pct:+.4f}%)")
print(f"MLP hidden dimension         : {MLP_HIDDEN_DIM}")
print("=" * 70)

# Sanity assertion
assert abs(parameter_difference_pct) < 0.1, "Parameter count is not sufficiently matched."
print("✅ Parameter count successfully matched to vanilla PMT within 0.1%.")


🔢 PARAMETER-MATCHING VERIFICATION
CEFR PT trainable parameters : 536,698,384 (536.698M)
Vanilla PMT reference        : 536,870,912 (536.871M)
Difference                   : -172,528 (-0.0321%)
MLP hidden dimension         : 7696
✅ Parameter count successfully matched to vanilla PMT within 0.1%.


In [ ]:
# =========================================================
# 10. DATASET UTILITY (BLIND INSTRUCTION PROMPT)
# =========================================================
class CEFRTopicDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length=512):
        self.data = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        topic_title = str(row["topic_title"]).strip()

        # Blind prompt: No explicit target CEFR level is supplied.
        prompt = (
            "You are an expert English language teacher demonstrating CEFR proficiency levels. "
            "Your task is to write a flawless, grammatically correct text responding to "
            f"this prompt: '{topic_title}'. "
            "If the requested target level is A1/A2, use very simple vocabulary, short sentences, "
            "and primitive structures. "
            "If the requested target level is C1/C2, utilize highly advanced vocabulary, idioms, "
            "and complex sentence patterns. "
            "Write only the direct response. Do not write any meta-commentary, greetings, "
            "or conversational pleasantries."
        )

        messages = [{"role": "user", "content": prompt}]
        formatted_prompt = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        full_text = formatted_prompt + str(row["clean_text"]).strip() + self.tokenizer.eos_token

        prompt_enc = self.tokenizer(
            formatted_prompt,
            add_special_tokens=False,
            truncation=True,
            max_length=self.max_length
        )
        prompt_len = len(prompt_enc["input_ids"])

        full_enc = self.tokenizer(
            full_text,
            add_special_tokens=False,
            truncation=True,
            max_length=self.max_length,
            padding="max_length"
        )

        input_ids = torch.tensor(full_enc["input_ids"], dtype=torch.long)
        attention_mask = torch.tensor(full_enc["attention_mask"], dtype=torch.long)

        # Mask prompt + padding
        labels = input_ids.clone()
        labels[:prompt_len] = -100
        labels[attention_mask == 0] = -100

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels,
            "cefr_id": torch.tensor(row["cefr_id"], dtype=torch.long)
        }


# =========================================================
# 11. DATA LOADERS
# =========================================================
TRAIN_BATCH_SIZE = 16
GRADIENT_ACCUMULATION_STEPS = 1
VAL_BATCH_SIZE = 16

train_loader = DataLoader(
    CEFRTopicDataset(train_df, tokenizer),
    batch_size=TRAIN_BATCH_SIZE,
    shuffle=True,
    pin_memory=True
)

val_loader = DataLoader(
    CEFRTopicDataset(val_df, tokenizer),
    batch_size=VAL_BATCH_SIZE,
    shuffle=False,
    pin_memory=True
)

print("\n📦 DataLoader configuration")
print(f"Training micro-batch      : {TRAIN_BATCH_SIZE}")
print(f"Gradient accumulation     : {GRADIENT_ACCUMULATION_STEPS}")
print(f"Effective training batch  : {TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}")
print(f"Validation batch          : {VAL_BATCH_SIZE}")


📦 DataLoader configuration
Training micro-batch      : 16
Gradient accumulation     : 1
Effective training batch  : 16
Validation batch          : 16


In [ ]:
# =========================================================
# 12. HARDWARE PROFILER
# =========================================================
class GPUMonitor(threading.Thread):
    def __init__(self, delay=1.0):
        super(GPUMonitor, self).__init__()
        self.stopped = False
        self.delay = delay
        self.utilization_rates = []
        pynvml.nvmlInit()
        self.handle = pynvml.nvmlDeviceGetHandleByIndex(0)

    def run(self):
        while not self.stopped:
            try:
                util = pynvml.nvmlDeviceGetUtilizationRates(self.handle)
                self.utilization_rates.append(util.gpu)
            except Exception:
                pass
            time.sleep(self.delay)

    def stop(self):
        self.stopped = True
        try:
            pynvml.nvmlShutdown()
        except Exception:
            pass

    def get_avg_utilization(self):
        if not self.utilization_rates: return 0.0
        return sum(self.utilization_rates) / len(self.utilization_rates)


# =========================================================
# 13. OPTIMIZATION & TRAINING CONFIGURATION
# =========================================================
epochs = 3
lr = 2e-4

optimizer = torch.optim.AdamW(prefix_controller.parameters(), lr=lr, weight_decay=0.01)

total_training_steps = math.ceil(len(train_loader) / GRADIENT_ACCUMULATION_STEPS) * epochs
warmup_steps = int(0.05 * total_training_steps)

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_training_steps
)

print("\n⚙️ Training configuration")
print(f"Epochs          : {epochs}")
print(f"Learning rate   : {lr}")
print(f"Warmup steps    : {warmup_steps}")
print(f"Training steps  : {total_training_steps}")


⚙️ Training configuration
Epochs          : 3
Learning rate   : 0.0002
Warmup steps    : 47
Training steps  : 942


In [ ]:
# =========================================================
# 14. TRAINING LOOP
# =========================================================
best_val_loss = float("inf")
training_summary_logs = []

print("\n" + "=" * 70)
print("🚀 Launching PARAMETER-MATCHED CEFR PREFIX-TUNING")
print("=" * 70)

torch.cuda.reset_peak_memory_stats()
gpu_monitor = GPUMonitor(delay=1.0)
gpu_monitor.start()

start_time = time.perf_counter()

for epoch in range(epochs):

    # =====================================================
    # TRAIN
    # =====================================================
    prefix_controller.train()
    running_train_loss = 0.0
    optimizer.zero_grad()

    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch + 1}/{epochs} [Train]", leave=False)

    for step, batch in enumerate(progress_bar):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        cefr_ids = batch["cefr_id"].to(device)

        # 1. CEFR-specific prefix
        past_key_values = prefix_controller(cefr_ids)

        # 2. Prefix attention mask
        prefix_mask = torch.ones(
            input_ids.shape[0],
            prefix_controller.num_virtual_tokens,
            dtype=attention_mask.dtype,
            device=device
        )

        full_attention_mask = torch.cat([prefix_mask, attention_mask], dim=1)

        # 3. Position IDs
        position_ids = full_attention_mask.long().cumsum(-1) - 1
        position_ids.masked_fill_(full_attention_mask == 0, 1)
        position_ids = position_ids[:, prefix_controller.num_virtual_tokens:]

        # 4. Frozen Llama forward
        outputs = base_model(
            input_ids=input_ids,
            attention_mask=full_attention_mask,
            position_ids=position_ids,
            past_key_values=past_key_values,
            labels=labels
        )

        loss = outputs.loss
        loss_for_backward = loss / GRADIENT_ACCUMULATION_STEPS
        loss_for_backward.backward()

        running_train_loss += loss.item()

        # Optimizer step
        if ((step + 1) % GRADIENT_ACCUMULATION_STEPS == 0) or ((step + 1) == len(train_loader)):
            torch.nn.utils.clip_grad_norm_(prefix_controller.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

        progress_bar.set_postfix({"Loss": f"{loss.item():.4f}"})

    # =====================================================
    # VALIDATION
    # =====================================================
    prefix_controller.eval()
    val_loss_accum = 0.0
    val_steps = 0

    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f"Epoch {epoch + 1}/{epochs} [Val]", leave=False):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            cefr_ids = batch["cefr_id"].to(device)

            past_key_values = prefix_controller(cefr_ids)

            prefix_mask = torch.ones(
                input_ids.shape[0],
                prefix_controller.num_virtual_tokens,
                dtype=attention_mask.dtype,
                device=device
            )

            full_attention_mask = torch.cat([prefix_mask, attention_mask], dim=1)

            position_ids = full_attention_mask.long().cumsum(-1) - 1
            position_ids.masked_fill_(full_attention_mask == 0, 1)
            position_ids = position_ids[:, prefix_controller.num_virtual_tokens:]

            outputs = base_model(
                input_ids=input_ids,
                attention_mask=full_attention_mask,
                position_ids=position_ids,
                past_key_values=past_key_values,
                labels=labels
            )

            val_loss_accum += outputs.loss.item()
            val_steps += 1

    # =====================================================
    # EPOCH METRICS
    # =====================================================
    avg_train_loss = running_train_loss / len(train_loader)
    avg_val_loss = val_loss_accum / val_steps
    val_ppl = float(np.exp(avg_val_loss))

    epoch_metrics = {
        "epoch": epoch + 1,
        "train_loss": round(avg_train_loss, 4),
        "val_loss": round(avg_val_loss, 4),
        "val_ppl": round(val_ppl, 2)
    }

    training_summary_logs.append(epoch_metrics)

    print(f"\n📈 [Epoch {epoch + 1}/{epochs}] Done | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val PPL: {val_ppl:.2f}")

    # =====================================================
    # SAVE BEST CHECKPOINT
    # =====================================================
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss

        checkpoint_path = os.path.join(LOCAL_OUTPUT_DIR, "cefr_prefix_tuning_537m_best_weights.pt")
        torch.save(prefix_controller.state_dict(), checkpoint_path)

        print("🌟 Best validation score achieved!")
        print(f"Saved checkpoint to: {checkpoint_path}")

    torch.cuda.empty_cache()
    gc.collect()





🚀 Launching PARAMETER-MATCHED CEFR PREFIX-TUNING


Epoch 1/3 [Train]:   0%|          | 0/314 [00:00<?, ?it/s]

Epoch 1/3 [Val]:   0%|          | 0/35 [00:00<?, ?it/s]


📈 [Epoch 1/3] Done | Train Loss: 2.6481 | Val Loss: 2.6245 | Val PPL: 13.80
🌟 Best validation score achieved!
Saved checkpoint to: /content/cefr_pt_parameter_matched_537m_output/cefr_prefix_tuning_537m_best_weights.pt


Epoch 2/3 [Train]:   0%|          | 0/314 [00:00<?, ?it/s]

Epoch 2/3 [Val]:   0%|          | 0/35 [00:00<?, ?it/s]


📈 [Epoch 2/3] Done | Train Loss: 2.4545 | Val Loss: 2.4497 | Val PPL: 11.58
🌟 Best validation score achieved!
Saved checkpoint to: /content/cefr_pt_parameter_matched_537m_output/cefr_prefix_tuning_537m_best_weights.pt


Epoch 3/3 [Train]:   0%|          | 0/314 [00:00<?, ?it/s]

Epoch 3/3 [Val]:   0%|          | 0/35 [00:00<?, ?it/s]


📈 [Epoch 3/3] Done | Train Loss: 2.3074 | Val Loss: 2.4371 | Val PPL: 11.44
🌟 Best validation score achieved!
Saved checkpoint to: /content/cefr_pt_parameter_matched_537m_output/cefr_prefix_tuning_537m_best_weights.pt


In [ ]:
# =========================================================
# 15. END PROFILING
# =========================================================
gpu_monitor.stop()
end_time = time.perf_counter()

total_time_seconds = end_time - start_time
total_time_hours = total_time_seconds / 3600
avg_gpu_util = gpu_monitor.get_avg_utilization()
peak_vram_gb = torch.cuda.max_memory_allocated() / (1024 ** 3)

print("\n" + "=" * 70)
print("🏁 TRAINING COMPLETE")
print("=" * 70)
print(f"Total training time: {total_time_seconds:.2f} seconds ({total_time_hours:.2f} hours)")
print(f"Peak GPU memory: {peak_vram_gb:.2f} GB")
print(f"Average GPU utilization: {avg_gpu_util:.1f}%")
print(f"Trainable parameters: {trainable_params:,}")
print("=" * 70)



🏁 TRAINING COMPLETE
Total training time: 1516.84 seconds (0.42 hours)
Peak GPU memory: 70.30 GB
Average GPU utilization: 97.4%
Trainable parameters: 536,698,384


In [ ]:
# =========================================================
# 16. SAVE TRAINING SUMMARY
# =========================================================
json_log_filename = "cefr_prefix_tuning_training_summary_537m_parameter_matched.json"

json_log_path = os.path.join("/content/drive/MyDrive/Mohammd_Thesis/Training_Logs/cefr_pt_parameter_matched_537m", json_log_filename)
with open(json_log_path, "w") as f:
    json.dump(training_summary_logs, f, indent=4)

local_summary_path = os.path.join(LOCAL_OUTPUT_DIR, json_log_filename)
with open(local_summary_path, "w") as f:
    json.dump(training_summary_logs, f, indent=4)

In [ ]:
DRIVE_LOG_DIR = "/content/drive/MyDrive/Mohammd_Thesis/Training_Logs/cefr_pt_parameter_matched_537m"

In [ ]:
# =========================================================
# 17. SAVE PARAMETER / HARDWARE PROFILE
# =========================================================
profiling_report = {
    "experiment_name": EXPERIMENT_NAME,
    "base_model": model_id,
    "conditioning": "6-way CEFR prefix gating",
    "num_cefr_levels": 6,
    "num_virtual_tokens": 30,
    "mlp_hidden_dim": MLP_HIDDEN_DIM,
    "trainable_parameters": trainable_params,
    "trainable_parameters_million": trainable_params / 1e6,
    "vanilla_pmt_reference_parameters": PMT_REFERENCE_PARAMS,
    "parameter_difference": parameter_difference,
    "parameter_difference_percent": parameter_difference_pct,
    "total_training_time_seconds": total_time_seconds,
    "total_training_time_hours": total_time_hours,
    "peak_gpu_memory_gb": peak_vram_gb,
    "average_gpu_utilization_percent": avg_gpu_util,
    "train_batch_size": TRAIN_BATCH_SIZE,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "effective_batch_size": (TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS),
    "epochs": epochs,
    "learning_rate": lr,
    "seed": SEED
}

profiling_json_path = os.path.join(DRIVE_LOG_DIR, "cefr_pt_537m_parameter_matched_profiling.json")
with open(profiling_json_path, "w") as f:
    json.dump(profiling_report, f, indent=4)

print(f"\n📊 Profiling report saved to:\n{profiling_json_path}")


# =========================================================
# 18. SAVE TOKENIZER
# =========================================================
tokenizer.save_pretrained(LOCAL_OUTPUT_DIR)


📊 Profiling report saved to:
/content/drive/MyDrive/Mohammd_Thesis/Training_Logs/cefr_pt_parameter_matched_537m/cefr_pt_537m_parameter_matched_profiling.json


('/content/cefr_pt_parameter_matched_537m_output/tokenizer_config.json',
 '/content/cefr_pt_parameter_matched_537m_output/chat_template.jinja',
 '/content/cefr_pt_parameter_matched_537m_output/tokenizer.json')

In [ ]:
# =========================================================
# 19. SAVE EXPERIMENT METADATA
# =========================================================
metadata = {
    "experiment_name": EXPERIMENT_NAME,
    "base_model": model_id,
    "method": "CEFR Multi-Prefix Prefix-Tuning",
    "parameter_matching": "Matched to vanilla PMT",
    "pmt_reference_parameters": PMT_REFERENCE_PARAMS,
    "trainable_parameters": trainable_params,
    "parameter_difference": parameter_difference,
    "parameter_difference_percent": parameter_difference_pct,
    "cefr_levels": ["A1", "A2", "B1", "B2", "C1", "C2"],
    "num_virtual_tokens_per_level": 30,
    "total_virtual_tokens": 180,
    "mlp_hidden_dimension": MLP_HIDDEN_DIM,
    "prefix_output_dimension": (
        base_model.config.num_hidden_layers
        * 2
        * base_model.config.num_key_value_heads
        * getattr(
            base_model.config,
            "head_dim",
            base_model.config.hidden_size // base_model.config.num_attention_heads
        )
    ),
    "conditioning_mechanism": "CEFR-specific prefix embedding selection",
    "textual_cefr_conditioning": "None",
    "backbone": "Frozen Llama-3.1-8B-Instruct",
    "dataset": "Balanced 6k CEFR dataset",
    "seed": SEED
}

metadata_path = os.path.join(LOCAL_OUTPUT_DIR, "experiment_metadata.json")
with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=4)

In [ ]:
# =========================================================
# 20. GENERATE README
# =========================================================

readme_content = f"""---
license: apache-2.0
base_model: {model_id}
tags:
- prefix-tuning
- cefr-control
- cefr-gating
- multi-prefix
- parameter-matched
- topic-aligned
---

# CEFR Prefix-Tuning — Parameter-Matched PMT Control (~537M)

This repository contains a parameter-matched CEFR Prefix-Tuning model based on Llama-3.1-8B-Instruct.

## Purpose

This experiment was designed as a controlled comparison against vanilla PrefixMemory-Tuning (PMT).

The primary purpose is to determine whether the performance improvement of the CEFR-conditioned model can be attributed primarily to its CEFR conditioning mechanism ("CEFR knob") rather than simply to a larger number of trainable parameters.

## Parameter Matching

| Model | Trainable Parameters |
|---|---:|
| Vanilla PMT | {PMT_REFERENCE_PARAMS:,} |
| This CEFR PT | {trainable_params:,} |

Parameter difference:
**{parameter_difference:+,} parameters ({parameter_difference_pct:+.4f}%)**

## CEFR Conditioning

Six independent CEFR prefix embedding sets are learned (A1 to C2).
Each level has 30 virtual prefix-token embeddings.

Therefore:
- 6 CEFR levels
- 30 virtual tokens per level
- 180 learned CEFR-specific prefix embeddings
- Embedding dimension: 4096

At inference time, the selected CEFR level determines which prefix embedding set is injected into the model.

## Prefix Reparameterization

The selected prefix embeddings are transformed using a shared two-layer MLP:

4096 → {MLP_HIDDEN_DIM} → 65536

The resulting representation is reshaped into layer-wise key/value states for the 32 Transformer layers.

## Backbone

Frozen: `{model_id}`

## Training

Dataset: Balanced 6k CEFR dataset.

The textual prompt does not explicitly provide the target CEFR level. CEFR conditioning is provided through the learned continuous prefix mechanism.

## Training Configuration

- **Epochs**: {epochs}
- **Learning rate**: {lr}
- **Training batch size**: {TRAIN_BATCH_SIZE}
- **Gradient accumulation**: {GRADIENT_ACCUMULATION_STEPS}
- **Effective batch size**: {TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}
- **Optimizer**: AdamW
- **Weight decay**: 0.01
- **Scheduler**: cosine with warmup
- **Warmup**: 5%
- **Seed**: {SEED}

## Training Results

{json.dumps(training_summary_logs, indent=2)}

## Hardware Profiling

- **Total training time**: {total_time_seconds:.2f} seconds ({total_time_hours:.2f} hours)
- **Peak GPU memory**: {peak_vram_gb:.2f} GB
- **Average GPU utilization**: {avg_gpu_util:.1f}%

## Experiment Role

This model is intended as a parameter-matched control experiment for the thesis evaluation of CEFR-conditioned Prefix-Tuning.

It should be compared against:

1. Standard Prefix-Tuning (~68M in the reported PEFT baseline, if applicable)
2. Previous CEFR Prefix-Tuning (~268M)
3. Parameter-matched CEFR Prefix-Tuning (~537M)
4. Vanilla PMT (~537M)
5. LoRA baseline (~42M)

The central experimental question is whether increasing the capacity of the CEFR Prefix-Tuning controller to approximately the same parameter budget as PMT closes the performance gap, or whether the CEFR conditioning mechanism remains the dominant factor.
"""

readme_path = os.path.join(LOCAL_OUTPUT_DIR, "README.md")

with open(readme_path, "w", encoding="utf-8") as f:
    f.write(readme_content)

print(f"\n✅ README saved to:\n{readme_path}")


✅ README saved to:
/content/cefr_pt_parameter_matched_537m_output/README.md


In [ ]:
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

from huggingface_hub import HfApi, login
from google.colab import userdata

LOCAL_OUTPUT_DIR = "/content/cefr_pt_parameter_matched_537m_output"
HF_REPO_ID = "MohammadKhosravi/llama3.1-8b-cefr-pt-537m-param-matched"

# Authenticate via Colab secret or manual prompt
try:
    hf_token = userdata.get("HF_TOKEN")
    login(token=hf_token)
except Exception:
    login()

api = HfApi()

# Create repository if it does not exist
api.create_repo(
    repo_id=HF_REPO_ID,
    repo_type="model",
    exist_ok=True
)

# Upload entire directory contents
api.upload_folder(
    folder_path=LOCAL_OUTPUT_DIR,
    repo_id=HF_REPO_ID,
    repo_type="model",
    commit_message="Upload parameter-matched CEFR Prefix-Tuning (~537M) weights and configs"
)

print(f"\n🚀 Successfully pushed assets to:\nhttps://huggingface.co/{HF_REPO_ID}")


🚀 Successfully pushed assets to:
https://huggingface.co/MohammadKhosravi/llama3.1-8b-cefr-pt-537m-param-matched


## CEFR Multi-Prefix Generation and Evaluation

This section loads the trained ~537M CEFR Multi-Prefix controller and evaluates it on the final In-Domain Evaluation Prompt Matrix.

The textual prompt does not explicitly contain the requested CEFR class. Instead, the target proficiency level is supplied through the selected CEFR-specific prefix bank.

In [ ]:
# =========================================================
# 0. ENVIRONMENT SETUP
# =========================================================

!pip install -q transformers torch datasets tqdm huggingface_hub accelerate scikit-learn textstat spacy hf_transfer "torchao>=0.16.0"
!python -m spacy download en_core_web_sm

import os

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

import gc
import math
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import spacy
import textstat
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    mean_absolute_error
)

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    AutoModelForSequenceClassification,
    set_seed
)

from transformers.cache_utils import DynamicCache

from huggingface_hub import login, hf_hub_download
from google.colab import drive, userdata
from tqdm.auto import tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.1/177.1 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 133.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 129.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 101.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 134.1 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


/usr/local/lib/python3.13/dist-packages/huggingface_hub/constants.py:299: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(


In [ ]:
# =========================================================
# 1. REPRODUCIBILITY
# =========================================================

set_seed(42)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


# =========================================================
# 2. MOUNT GOOGLE DRIVE
# =========================================================
#
# IMPORTANT:
# If /content/drive already exists and contains files,
# drive.mount() may throw:
#
#   ValueError: Mountpoint must not already contain files
#
# In that case, use:
#   drive.mount('/content/drive', force_remount=True)
#
# =========================================================

drive.mount('/content/drive', force_remount=True)


# =========================================================
# 3. EXPERIMENT PATHS
# =========================================================
#
# Completely separate from the previous 268M experiment.
# Nothing here overwrites the 268M results.
# =========================================================

INPUT_CSV = (
    "/content/drive/MyDrive/Your_Path/"
    "in_domain_evaluation_prompt_matrix.csv"
)

OUTPUT_DIR = (
    "/content/drive/MyDrive/Your_Path/"
    "cefr_prefix_tuning_537m_evaluation_results"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

OUTPUT_CSV_PATH = os.path.join(
    OUTPUT_DIR,
    "cefr_prefix_537m_benchmark_results_log.csv"
)

OUTPUT_TXT_PATH = os.path.join(
    OUTPUT_DIR,
    "cefr_prefix_537m_overall_metrics_report.txt"
)

OUTPUT_IMG_PATH = os.path.join(
    OUTPUT_DIR,
    "cefr_prefix_537m_confusion_matrix.png"
)


Mounted at /content/drive


In [ ]:
# =========================================================
# 4. HUGGING FACE AUTHENTICATION
# =========================================================

try:
    hf_token = userdata.get("HF_TOKEN")
    login(token=hf_token)
    print("✓ Hugging Face authentication successful.")
except Exception as e:
    print(f"⚠️ Hugging Face login skipped: {e}")


# =========================================================
# 5. GLOBAL CONFIGURATION
# =========================================================

device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Device: {device}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(
        f"GPU memory: "
        f"{torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB"
    )


label_map = {
    "A1": 0,
    "A2": 1,
    "B1": 2,
    "B2": 3,
    "C1": 4,
    "C2": 5
}

inv_label_map = {
    v: k for k, v in label_map.items()
}

✓ Hugging Face authentication successful.
Device: cuda
GPU: NVIDIA A100-SXM4-80GB
GPU memory: 79.25 GB


In [ ]:
# =========================================================
# 6. MODEL CONFIGURATION
# =========================================================

model_id = "meta-llama/Llama-3.1-8B-Instruct"

HF_REPO_ID = (
    "MohammadKhosravi/"
    "llama3.1-8b-cefr-pt-537m-param-matched"
)

NUM_VIRTUAL_TOKENS = 30
NUM_CLASSES = 6
MLP_HIDDEN_DIM = 7696

BATCH_SIZE = 32

MAX_NEW_TOKENS = 200
TEMPERATURE = 0.6

print("\n=========================================================")
print("CEFR PT 537M PARAMETER-MATCHED INFERENCE")
print("=========================================================")
print(f"Base model          : {model_id}")
print(f"Controller          : {HF_REPO_ID}")
print(f"Virtual tokens      : {NUM_VIRTUAL_TOKENS}")
print(f"CEFR classes        : {NUM_CLASSES}")
print(f"MLP hidden dimension: {MLP_HIDDEN_DIM}")
print(f"Generation batch    : {BATCH_SIZE}")
print(f"Max new tokens      : {MAX_NEW_TOKENS}")
print(f"Temperature         : {TEMPERATURE}")
print("=========================================================\n")


# =========================================================
# 7. LOAD TOKENIZER
# =========================================================

print("[1/5] Loading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    model_id,
    padding_side="left"
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

print("✓ Tokenizer loaded.")


# =========================================================
# 8. LOAD FROZEN LLAMA-3.1-8B
# =========================================================

print("\n[2/5] Loading Llama-3.1-8B-Instruct...")

base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

base_model.eval()

# Generation requires cache.
base_model.config.use_cache = True

print("✓ Base model loaded.")
print("✓ Base model frozen.")


CEFR PT 537M PARAMETER-MATCHED INFERENCE
Base model          : meta-llama/Llama-3.1-8B-Instruct
Controller          : MohammadKhosravi/llama3.1-8b-cefr-pt-537m-param-matched
Virtual tokens      : 30
CEFR classes        : 6
MLP hidden dimension: 7696
Generation batch    : 32
Max new tokens      : 200
Temperature         : 0.6

[1/5] Loading tokenizer...


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

✓ Base model loaded.
✓ Base model frozen.


In [ ]:
# =========================================================
# 9. EXACT 537M TRAINING CONTROLLER
# =========================================================

class CEFRMultiPrefixController(nn.Module):
    """
    EXACT controller architecture used during the 537M
    parameter-matched CEFR Prefix-Tuning training run.

    Architecture:

        6 CEFR levels
            |
            v
        30 dedicated virtual tokens / level
            |
            v
        4096-dimensional prefix embeddings
            |
            v
        Linear(4096 -> 7696)
            |
            v
        Tanh
            |
            v
        Linear(7696 -> layer-wise KV dimension)
            |
            v
        DynamicCache

    The six CEFR levels share the MLP but have independent
    prefix embeddings.
    """

    def __init__(
        self,
        config,
        num_virtual_tokens=30,
        num_classes=6,
        mlp_hidden_dim=7696
    ):
        super().__init__()

        self.num_virtual_tokens = num_virtual_tokens
        self.num_classes = num_classes

        self.num_layers = config.num_hidden_layers
        self.hidden_size = config.hidden_size

        self.num_kv_heads = config.num_key_value_heads

        self.head_dim = getattr(
            config,
            "head_dim",
            config.hidden_size // config.num_attention_heads
        )

        self.mlp_hidden_dim = mlp_hidden_dim

        # -------------------------------------------------
        # Six independent CEFR prefix embedding groups
        #
        # 6 × 30 × 4096
        # -------------------------------------------------

        self.prefix_embeddings = nn.Embedding(
            num_classes * num_virtual_tokens,
            self.hidden_size
        )

        # -------------------------------------------------
        # Layer-wise K/V output dimension
        # -------------------------------------------------

        flat_out_dim = (
            self.num_layers
            * 2
            * self.num_kv_heads
            * self.head_dim
        )

        # -------------------------------------------------
        # High-capacity shared MLP
        # -------------------------------------------------

        self.prefix_mlp = nn.Sequential(

            nn.Linear(
                self.hidden_size,
                self.mlp_hidden_dim
            ),

            nn.Tanh(),

            nn.Linear(
                self.mlp_hidden_dim,
                flat_out_dim
            )
        )

    def forward(self, cefr_ids):

        batch_size = cefr_ids.shape[0]

        # -------------------------------------------------
        # Select the 30 virtual tokens corresponding to
        # the requested CEFR level.
        #
        # A1 -> indices 0..29
        # A2 -> indices 30..59
        # ...
        # C2 -> indices 150..179
        # -------------------------------------------------

        class_offsets = (
            cefr_ids * self.num_virtual_tokens
        ).unsqueeze(1)

        base_indices = torch.arange(
            self.num_virtual_tokens,
            device=cefr_ids.device
        ).unsqueeze(0)

        token_indices = (
            class_offsets + base_indices
        )

        # [B, 30, 4096]

        prefix_tokens = self.prefix_embeddings(
            token_indices
        )

        # [B, 30, flat_out_dim]

        past_kv_flat = self.prefix_mlp(
            prefix_tokens
        )

        # -------------------------------------------------
        # Reshape:
        #
        # [B, 30, layers, 2, KV_heads, head_dim]
        # -------------------------------------------------

        past_kv = past_kv_flat.view(
            batch_size,
            self.num_virtual_tokens,
            self.num_layers,
            2,
            self.num_kv_heads,
            self.head_dim
        )

        # -------------------------------------------------
        # [layers, 2, B, KV_heads, 30, head_dim]
        # -------------------------------------------------

        past_kv = past_kv.permute(
            2,
            3,
            0,
            4,
            1,
            5
        )

        # -------------------------------------------------
        # Build Hugging Face DynamicCache
        # -------------------------------------------------

        past_key_values = DynamicCache()

        for i in range(self.num_layers):

            past_key_values.update(
                past_kv[i, 0],
                past_kv[i, 1],
                layer_idx=i
            )

        return past_key_values

In [ ]:
# =========================================================
# 10. INITIALIZE CONTROLLER
# =========================================================

print("\nInitializing 537M CEFR controller...")

prefix_controller = CEFRMultiPrefixController(
    base_model.config,
    num_virtual_tokens=NUM_VIRTUAL_TOKENS,
    num_classes=NUM_CLASSES,
    mlp_hidden_dim=MLP_HIDDEN_DIM
).to(
    device,
    dtype=torch.bfloat16
)

trainable_params = sum(
    p.numel()
    for p in prefix_controller.parameters()
    if p.requires_grad
)

print(
    f"\nController trainable parameters:"
    f" {trainable_params:,}"
)

print(
    f"Controller trainable parameters:"
    f" {trainable_params / 1e6:.2f}M"
)


Initializing 537M CEFR controller...

Controller trainable parameters: 536,698,384
Controller trainable parameters: 536.70M


In [ ]:
# =========================================================
# 11. LOAD TRAINED CONTROLLER WEIGHTS
# =========================================================

print(
    f"\nDownloading trained controller from:\n"
    f"{HF_REPO_ID}"
)

weights_path = hf_hub_download(
    repo_id=HF_REPO_ID,
    filename="cefr_prefix_tuning_537m_best_weights.pt"
)

print(f"✓ Weights downloaded to:\n{weights_path}")


# ---------------------------------------------------------
# IMPORTANT:
# Loading directly into the same architecture ensures that
# the inference controller is identical to the training
# controller.
# ---------------------------------------------------------

state_dict = torch.load(
    weights_path,
    map_location=device
)

prefix_controller.load_state_dict(
    state_dict,
    strict=True
)

prefix_controller.eval()

print("✓ Controller weights loaded successfully.")
print("✓ Strict state-dict match confirmed.")


MohammadKhosravi/llama3.1-8b-cefr-pt-537m-param-matched


cefr_prefix_tuning_537m_best_weights.pt: reconstructing file:   0%|          |  0.00B / 1.07GB            

cefr_prefix_tuning_537m_best_weights.pt: downloading bytes:           |  0.00B            

✓ Weights downloaded to:
/root/.cache/huggingface/hub/models--MohammadKhosravi--llama3.1-8b-cefr-pt-537m-param-matched/snapshots/12e815db357168b763533455a10cd74c12c63cc4/cefr_prefix_tuning_537m_best_weights.pt
✓ Controller weights loaded successfully.
✓ Strict state-dict match confirmed.


In [ ]:
# =========================================================
# 12. VERIFY PARAMETER COUNT
# =========================================================

loaded_params = sum(
    p.numel()
    for p in prefix_controller.parameters()
)

print(
    f"\n✓ Verified controller parameter count:"
    f" {loaded_params:,}"
)

print(
    f"✓ Verified controller size:"
    f" {loaded_params / 1e6:.2f}M"
)


# =========================================================
# 13. LOAD BENCHMARK
# =========================================================

print("\n[3/5] Loading benchmark dataset...")

df = pd.read_csv(INPUT_CSV)

print(f"Benchmark samples: {len(df):,}")

required_columns = [
    "topic_id",
    "topic_title",
    "cefr"
]

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

df["cefr"] = (
    df["cefr"]
    .astype(str)
    .str.strip()
    .str.upper()
)

invalid_levels = sorted(
    set(df["cefr"]) - set(label_map.keys())
)

if invalid_levels:
    raise ValueError(
        f"Invalid CEFR labels found: {invalid_levels}"
    )

print("✓ Benchmark validated.")


# =========================================================
# 14. BLIND PROMPT
# =========================================================
#
# EXACT SAME PROMPT AS TRAINING.
#
# The target CEFR level is NOT inserted into the text.
#
# The CEFR level is supplied exclusively through the
# controller.
# =========================================================

def build_blind_prompt(topic_title):

    prompt = (
        f"You are an expert English language teacher "
        f"demonstrating CEFR proficiency levels. "
        f"Your task is to write a flawless, grammatically "
        f"correct text responding to this prompt: "
        f"'{topic_title}'. "
        f"If the requested target level is A1/A2, use very "
        f"simple vocabulary, short sentences, and primitive "
        f"structures. "
        f"If the requested target level is C1/C2, utilize "
        f"highly advanced vocabulary, idioms, and complex "
        f"sentence patterns. "
        f"Write only the direct response. Do not write any "
        f"meta-commentary, greetings, or conversational "
        f"pleasantries."
    )

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )


✓ Verified controller parameter count: 536,698,384
✓ Verified controller size: 536.70M

[3/5] Loading benchmark dataset...
Benchmark samples: 702
✓ Benchmark validated.


In [ ]:
# =========================================================
# 15. PREFIX-AWARE AUTOREGRESSIVE GENERATION
# =========================================================
#
# This is the critical part.
#
# We DO NOT prepend dummy tokens.
#
# The controller creates 30 virtual K/V states in the
# DynamicCache.
#
# The actual prompt is then passed as the unprocessed
# sequence.
#
# This follows the semantics of the training implementation
# and Hugging Face's Cache API.
# =========================================================

def generate_with_cefr_prefix(
    input_ids,
    attention_mask,
    cefr_ids,
    max_new_tokens=200,
    temperature=0.6
):

    batch_size = input_ids.shape[0]

    num_virtual_tokens = (
        prefix_controller.num_virtual_tokens
    )

    # -----------------------------------------------------
    # STEP 1
    # Generate CEFR-specific prefix cache.
    #
    # Exactly the same controller call as training.
    # -----------------------------------------------------

    past_key_values = prefix_controller(
        cefr_ids
    )

    # -----------------------------------------------------
    # STEP 2
    # Build attention mask exactly like training.
    #
    # [prefix mask | actual prompt mask]
    # -----------------------------------------------------

    prefix_mask = torch.ones(
        batch_size,
        num_virtual_tokens,
        dtype=attention_mask.dtype,
        device=input_ids.device
    )

    full_attention_mask = torch.cat(
        [
            prefix_mask,
            attention_mask
        ],
        dim=1
    )

    # -----------------------------------------------------
    # STEP 3
    # Position IDs exactly follow training.
    #
    # This is important because the virtual prefix occupies
    # the first 30 cache positions.
    # -----------------------------------------------------

    position_ids = (
        full_attention_mask.long()
        .cumsum(-1)
        - 1
    )

    position_ids.masked_fill_(
        full_attention_mask == 0,
        1
    )

    # Remove prefix positions because the actual input
    # tokens are the unprocessed tokens.
    position_ids = (
        position_ids[
            :,
            num_virtual_tokens:
        ]
    )

    # -----------------------------------------------------
    # STEP 4
    # Prefill the cache with the actual prompt.
    #
    # IMPORTANT:
    # No dummy tokens are added.
    # -----------------------------------------------------

    outputs = base_model(
        input_ids=input_ids,
        attention_mask=full_attention_mask,
        position_ids=position_ids,
        past_key_values=past_key_values,
        use_cache=True
    )

    # -----------------------------------------------------
    # The cache has now been extended:
    #
    #   [30 virtual prefix tokens]
    #   +
    #   [prompt tokens]
    #
    # -----------------------------------------------------

    past_key_values = outputs.past_key_values

    # Last-token logits produce the first generated token.
    next_token_logits = outputs.logits[:, -1, :]

    generated_tokens = []

    # Track how many generated tokens have been produced.
    generated_count = 0

    finished = torch.zeros(
        batch_size,
        dtype=torch.bool,
        device=input_ids.device
    )

    # -----------------------------------------------------
    # STEP 5
    # Autoregressive decoding
    # -----------------------------------------------------

    for step in range(max_new_tokens):

        # Temperature sampling
        logits = (
            next_token_logits / temperature
        )

        probabilities = torch.softmax(
            logits,
            dim=-1
        )

        next_tokens = torch.multinomial(
            probabilities,
            num_samples=1
        ).squeeze(-1)

        # Keep EOS for finished sequences from generating
        # arbitrary additional content.
        next_tokens = torch.where(
            finished,
            torch.full_like(
                next_tokens,
                tokenizer.pad_token_id
            ),
            next_tokens
        )

        generated_tokens.append(
            next_tokens
        )

        generated_count += 1

        # Mark sequences that generated EOS.
        finished = (
            finished
            | (next_tokens == tokenizer.eos_token_id)
        )

        if torch.all(finished):
            break

        # -------------------------------------------------
        # Add the new token to the attention mask.
        # -------------------------------------------------

        new_token_mask = torch.ones(
            batch_size,
            1,
            dtype=attention_mask.dtype,
            device=input_ids.device
        )

        full_attention_mask = torch.cat(
            [
                full_attention_mask,
                new_token_mask
            ],
            dim=1
        )

        # -------------------------------------------------
        # Position ID of the new token.
        #
        # For every sequence, the position is the number
        # of active tokens before this token minus one.
        # -------------------------------------------------

        next_position_ids = (
            full_attention_mask.long()
            .cumsum(-1)
            - 1
        )

        next_position_ids.masked_fill_(
            full_attention_mask == 0,
            1
        )

        next_position_ids = (
            next_position_ids[:, -1:]
        )

        # -------------------------------------------------
        # Forward ONLY the newly generated token.
        #
        # The previous prefix + prompt + generated tokens
        # are already represented inside DynamicCache.
        # -------------------------------------------------

        outputs = base_model(
            input_ids=next_tokens.unsqueeze(1),
            attention_mask=full_attention_mask,
            position_ids=next_position_ids,
            past_key_values=past_key_values,
            use_cache=True
        )

        past_key_values = outputs.past_key_values

        next_token_logits = (
            outputs.logits[:, -1, :]
        )

    # -----------------------------------------------------
    # [B, generated_length]
    # -----------------------------------------------------

    if generated_tokens:

        generated_ids = torch.stack(
            generated_tokens,
            dim=1
        )

    else:

        generated_ids = torch.empty(
            batch_size,
            0,
            dtype=torch.long,
            device=input_ids.device
        )

    return generated_ids


# =========================================================
# 16. PHASE 1 — GENERATION
# =========================================================

print(
    "\n[4/5] Initiating Batch Generation Phase..."
)

generated_records = []

base_model.eval()
prefix_controller.eval()

for b_start in tqdm(
    range(0, len(df), BATCH_SIZE),
    desc="Processing Prompt Batches"
):

    batch_df = df.iloc[
        b_start:
        b_start + BATCH_SIZE
    ]

    batch_prompts = []
    batch_cefrs = []

    # -----------------------------------------------------
    # Construct exactly the same blind prompts as training.
    # -----------------------------------------------------

    for _, row in batch_df.iterrows():

        target_cefr = (
            str(row["cefr"])
            .strip()
            .upper()
        )

        batch_cefrs.append(
            target_cefr
        )

        batch_prompts.append(
            build_blind_prompt(
                row["topic_title"]
            )
        )

    # -----------------------------------------------------
    # Tokenization
    #
    # Left padding is used for batched autoregressive
    # generation so that the last column corresponds to
    # the actual prompt token.
    # -----------------------------------------------------

    inputs = tokenizer(
        batch_prompts,
        padding=True,
        return_tensors="pt"
    )

    input_ids = inputs.input_ids.to(
        device
    )

    attention_mask = inputs.attention_mask.to(
        device
    )

    cefr_tensor = torch.tensor(
        [
            label_map[c]
            for c in batch_cefrs
        ],
        dtype=torch.long,
        device=device
    )

    # -----------------------------------------------------
    # Generate
    # -----------------------------------------------------

    with torch.inference_mode():

        output_ids = generate_with_cefr_prefix(
            input_ids=input_ids,
            attention_mask=attention_mask,
            cefr_ids=cefr_tensor,
            max_new_tokens=MAX_NEW_TOKENS,
            temperature=TEMPERATURE
        )

    # -----------------------------------------------------
    # Decode each generated sequence.
    # -----------------------------------------------------

    for i, output_id in enumerate(output_ids):

        gen_text = tokenizer.decode(
            output_id,
            skip_special_tokens=True
        ).strip()

        generated_records.append(
            {
                "topic_id":
                    batch_df.iloc[i]["topic_id"],

                "topic_title":
                    batch_df.iloc[i]["topic_title"],

                "target_cefr":
                    batch_cefrs[i],

                "generated_text":
                    gen_text
            }
        )


# =========================================================
# 17. GENERATION SANITY CHECK
# =========================================================

print("\nGeneration completed.")

print(
    f"Total generated records:"
    f" {len(generated_records):,}"
)

print("\nExample generations:")
print("=" * 70)

for record in generated_records[:3]:

    print(
        f"\nTarget: {record['target_cefr']}"
    )

    print(
        f"Topic: {record['topic_title']}"
    )

    print(
        f"Generated:\n{record['generated_text']}"
    )

    print("-" * 70)


# =========================================================
# 18. FREE LLAMA + CONTROLLER MEMORY
# =========================================================

del base_model
del prefix_controller

torch.cuda.empty_cache()
gc.collect()

print(
    "\n✓ Base model and CEFR controller released."
)



[4/5] Initiating Batch Generation Phase...


Processing Prompt Batches:   0%|          | 0/22 [00:00<?, ?it/s]

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.



Generation completed.
Total generated records: 702

Example generations:

Target: A1
Topic: Taking inventory in the office
Generated:
There are a lot of people in the office. There is a lot of desks and chairs. There are a lot of computers and keyboards. There are a lot of notebooks and pens. There is a meeting room.
----------------------------------------------------------------------

Target: A2
Topic: Taking inventory in the office
Generated:
Hello, Mr. Smith. Here is the summary of my task. - I found a computer in the office. It was on the third floor. I took it to the computer room. - I found a chair in the office. It was on the third floor. I took it to the office. - I found a desk in the office. It was on the third floor. I took it to the office. - I found a vase in the office. It was on the third floor. I took it to the office. - I found a water bottle in the office. It was on the third floor. I took it to the office. - I found a fan in the office. It was on the third floor. 

In [ ]:
# =========================================================
# 19. PHASE 2 — EVALUATION
# =========================================================

print(
    "\n[5/5] Initiating Post-Generation Evaluation Loop..."
)


# =========================================================
# 20. LOAD SPACY
# =========================================================

nlp = spacy.load(
    "en_core_web_sm"
)


# =========================================================
# 21. LOAD JOINTLOSS CEFR JUDGE
# =========================================================

print(
    "Deploying Custom JointLoss RoBERTa Evaluator..."
)

JUDGE_MODEL_ID = (
    "MohammadKhosravi/"
    "roberta-large-cefr-classifier-JointLoss"
)

judge_tokenizer = AutoTokenizer.from_pretrained(
    JUDGE_MODEL_ID
)

judge_model = (
    AutoModelForSequenceClassification
    .from_pretrained(
        JUDGE_MODEL_ID
    )
    .to(device)
)

judge_model.eval()

print("✓ JointLoss evaluator loaded.")


# =========================================================
# 22. MEAN DEPENDENCY DISTANCE
# =========================================================

def calculate_mdd(text):

    doc = nlp(text)

    total_dist = 0
    tokens = 0

    for token in doc:

        if token.dep_ != "punct":

            total_dist += abs(
                token.i - token.head.i
            )

            tokens += 1

    if tokens > 0:
        return total_dist / tokens

    return 0.0


# =========================================================
# 23. SENTENCE DRIFT
# =========================================================

def calculate_sentence_drift(text):

    doc = nlp(text)

    sentences = [
        sent.text.strip()
        for sent in doc.sents
        if len(sent.text.strip()) > 10
    ]

    if len(sentences) <= 1:
        return 0

    inputs = judge_tokenizer(
        sentences,
        padding=True,
        truncation=True,
        max_length=512,
        return_tensors="pt"
    ).to(device)

    with torch.inference_mode():

        preds = torch.argmax(
            judge_model(**inputs).logits,
            dim=-1
        ).cpu().numpy()

    return int(
        np.max(preds) - np.min(preds)
    )


[5/5] Initiating Post-Generation Evaluation Loop...
Deploying Custom JointLoss RoBERTa Evaluator...


config.json:   0%|          | 0.00/900 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/388 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.56M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.42GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

✓ JointLoss evaluator loaded.


In [ ]:
# =========================================================
# 24. EVALUATE GENERATED TEXT
# =========================================================

final_results = []

for record in tqdm(
    generated_records,
    desc="Extracting Metrics"
):

    gen_text = record["generated_text"]

    target_cefr = record["target_cefr"]

    # -----------------------------------------------------
    # Prevent evaluator failure on empty generations.
    # -----------------------------------------------------

    if not gen_text:

        gen_text = (
            "Empty response generation failure."
        )

    # -----------------------------------------------------
    # CEFR classification
    # -----------------------------------------------------

    eval_inputs = judge_tokenizer(
        gen_text,
        padding=True,
        truncation=True,
        max_length=512,
        return_tensors="pt"
    ).to(device)

    with torch.inference_mode():

        eval_pred = torch.argmax(
            judge_model(
                **eval_inputs
            ).logits,
            dim=-1
        ).item()

    predicted_cefr = inv_label_map[
        eval_pred
    ]

    # -----------------------------------------------------
    # Strict accuracy
    # -----------------------------------------------------

    is_strict = int(
        predicted_cefr == target_cefr
    )

    # -----------------------------------------------------
    # Adjacent accuracy
    # -----------------------------------------------------

    is_adjacent = int(
        abs(
            label_map[predicted_cefr]
            -
            label_map[target_cefr]
        ) <= 1
    )

    # -----------------------------------------------------
    # Flesch Reading Ease
    # -----------------------------------------------------

    try:

        flesch_score = (
            textstat.flesch_reading_ease(
                gen_text
            )
        )

    except Exception:

        flesch_score = 0.0

    # -----------------------------------------------------
    # MDD
    # -----------------------------------------------------

    mdd_score = calculate_mdd(
        gen_text
    )

    # -----------------------------------------------------
    # Sentence drift
    # -----------------------------------------------------

    drift_score = calculate_sentence_drift(
        gen_text
    )

    # -----------------------------------------------------
    # Preserve original output schema.
    # -----------------------------------------------------

    final_results.append(
        {
            "topic_id":
                record["topic_id"],

            "topic_title":
                record["topic_title"],

            "target_cefr":
                target_cefr,

            "predicted_cefr":
                predicted_cefr,

            "strict_match":
                is_strict,

            "adjacent_match":
                is_adjacent,

            "mdd":
                round(mdd_score, 2),

            "readability_flesch":
                round(flesch_score, 2),

            "sentence_drift_max":
                drift_score,

            "generated_text":
                gen_text
        }
    )


Extracting Metrics:   0%|          | 0/702 [00:00<?, ?it/s]

In [ ]:
# =========================================================
# 25. SAVE BENCHMARK CSV
# =========================================================

df_final = pd.DataFrame(
    final_results
)

df_final.to_csv(
    OUTPUT_CSV_PATH,
    index=False
)

print(
    f"\n✔️ Benchmark log saved to:\n"
    f"{OUTPUT_CSV_PATH}"
)


# =========================================================
# 26. MACRO STATISTICS
# =========================================================

y_true = df_final[
    "target_cefr"
]

y_pred = df_final[
    "predicted_cefr"
]

strict_acc = (
    df_final["strict_match"].mean()
    * 100
)

adj_acc = (
    df_final["adjacent_match"].mean()
    * 100
)

mae = mean_absolute_error(
    y_true.map(label_map),
    y_pred.map(label_map)
)

avg_mdd = (
    df_final["mdd"].mean()
)

avg_flesch = (
    df_final["readability_flesch"].mean()
)

avg_drift = (
    df_final["sentence_drift_max"].mean()
)


# =========================================================
# 27. REPORT GENERATION
# =========================================================

report_text = "=" * 65 + "\n"

report_text += (
    " 📊 FINAL COMPILED MACRO-STATISTICS "
    "(CEFR PT ~537M PARAM-MATCHED)\n"
)

report_text += "=" * 65 + "\n"

report_text += (
    f"Total Processed       : "
    f"{len(df_final)}\n"
)

report_text += (
    f"Strict Accuracy       : "
    f"{strict_acc:.2f}%\n"
)

report_text += (
    f"Adjacent Accuracy     : "
    f"{adj_acc:.2f}%\n"
)

report_text += (
    f"Mean Abs Error (MAE)  : "
    f"{mae:.4f}\n"
)

report_text += (
    f"Avg MDD Score         : "
    f"{avg_mdd:.2f}\n"
)

report_text += (
    f"Avg Reading Ease      : "
    f"{avg_flesch:.2f}\n"
)

report_text += (
    f"Avg Sentence Drift    : "
    f"{avg_drift:.2f} levels\n"
)

report_text += (
    "Avg Perplexity        : "
    "(Computed externally)\n"
)

report_text += "=" * 65 + "\n\n"


# =========================================================
# 28. PER-LEVEL BREAKDOWN
# =========================================================

report_text += (
    "📈 PERFORMANCE BREAKDOWN BY CEFR TARGET LEVEL:\n"
)

level_agg = (
    df_final
    .groupby("target_cefr")
    .agg(
        strict_accuracy=(
            "strict_match",
            lambda x: np.mean(x) * 100
        ),

        avg_mdd=(
            "mdd",
            "mean"
        ),

        avg_flesch=(
            "readability_flesch",
            "mean"
        ),

        avg_drift=(
            "sentence_drift_max",
            "mean"
        )
    )
    .round(2)
)

report_text += (
    level_agg.to_string()
    + "\n\n"
)


# =========================================================
# 29. CLASSIFICATION REPORT
# =========================================================

report_text += "=" * 65 + "\n"

report_text += (
    "📝 DETAILED CLASSIFICATION REPORT:\n"
)

cefr_labels = [
    "A1",
    "A2",
    "B1",
    "B2",
    "C1",
    "C2"
]

report_text += classification_report(
    y_true,
    y_pred,
    labels=cefr_labels,
    zero_division=0
)

report_text += (
    "\n"
    + "=" * 65
    + "\n"
)


# =========================================================
# 30. SAVE TEXT REPORT
# =========================================================

with open(
    OUTPUT_TXT_PATH,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        report_text
    )

print(
    report_text
)


# =========================================================
# 31. CONFUSION MATRIX
# =========================================================

cm = confusion_matrix(
    y_true,
    y_pred,
    labels=cefr_labels
)

plt.figure(
    figsize=(8, 6)
)

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=cefr_labels,
    yticklabels=cefr_labels,
    cbar=True,
    square=True
)

plt.title(
    "CEFR Alignment "
    "(CEFR Multi-Prefix Tuning ~537M)",
    fontsize=12,
    pad=15
)

plt.xlabel(
    "Predicted CEFR Level "
    "(JointLoss Evaluator)",
    fontsize=10,
    labelpad=10
)

plt.ylabel(
    "Target CEFR Level "
    "(Dataset Input)",
    fontsize=10,
    labelpad=10
)

plt.tight_layout()

plt.savefig(
    OUTPUT_IMG_PATH,
    dpi=300
)

plt.close()

print(
    f"🎨 Confusion Matrix plot successfully saved to:\n"
    f"{OUTPUT_IMG_PATH}"
)


# =========================================================
# 32. FINAL SUMMARY
# =========================================================

print("\n")
print("=" * 70)
print("                    EXPERIMENT COMPLETE")
print("=" * 70)

print(
    f"Model                  : "
    f"CEFR PT ~537M"
)

print(
    f"Total benchmark        : "
    f"{len(df_final)}"
)

print(
    f"Strict accuracy        : "
    f"{strict_acc:.2f}%"
)

print(
    f"Adjacent accuracy      : "
    f"{adj_acc:.2f}%"
)

print(
    f"MAE                    : "
    f"{mae:.4f}"
)

print(
    f"Average MDD            : "
    f"{avg_mdd:.2f}"
)

print(
    f"Average Flesch         : "
    f"{avg_flesch:.2f}"
)

print(
    f"Average sentence drift: "
    f"{avg_drift:.2f}"
)

print("-" * 70)

print(
    f"CSV:\n{OUTPUT_CSV_PATH}"
)

print(
    f"Report:\n{OUTPUT_TXT_PATH}"
)

print(
    f"Confusion matrix:\n{OUTPUT_IMG_PATH}"
)

print("=" * 70)


✔️ Benchmark log saved to:
/content/drive/MyDrive/Mohammd_Thesis/Results/CEFR_Prefix_Tuning_537m_Param_Matched/cefr_prefix_537m_benchmark_results_log.csv
 📊 FINAL COMPILED MACRO-STATISTICS (CEFR PT ~537M PARAM-MATCHED)
Total Processed       : 702
Strict Accuracy       : 64.10%
Adjacent Accuracy     : 81.48%
Mean Abs Error (MAE)  : 0.6966
Avg MDD Score         : 1.80
Avg Reading Ease      : 81.22
Avg Sentence Drift    : 2.27 levels
Avg Perplexity        : (Computed externally)

📈 PERFORMANCE BREAKDOWN BY CEFR TARGET LEVEL:
             strict_accuracy  avg_mdd  avg_flesch  avg_drift
target_cefr                                                 
A1                     79.49     1.47       87.23       0.86
A2                     71.79     1.74       86.14       2.03
B1                     73.50     1.88       78.90       2.10
B2                     70.09     1.87       79.03       2.38
C1                     46.15     1.94       76.43       2.93
C2                     43.59     1.92       